In [2]:
from pathlib import Path 
from subprocess import run
import os
import sys

# File Permissions on macOS

All operating systems derived from Unix have a system for specifying the &ldquo;privileges&rdquo; that limit access to any file. The macOS operating system, which is based on BSD Unix, has such a system. This notebook explains how to use it. In particular, it shows how you can use file permissions to protect an ssh key. 

Windows uses a different system to decide who can read a file or write to it. But even if your personal workstation runs Windows, it is worth learning about file permissions because you may need to work on cloud servers that run Linux. 

In [76]:
if sys.platform == "darwin":
    print("You are working on a computer that runs macOS.\n")
    print("You will be able to run all the code in this notebook.")
elif sys.platform == "win32":
    print("You are working on a computer that runs Windows.")
    print("You will not be able to run some of the code in this notebook.")

You are working on a computer that runs macOS.

You will be able to run all the code in this notebook.


# Changing file permissions on macOS

You may have worked through the [ssh_keys notebook](ssh_keys.ipynb). If so, it used a function to create some keys for you in the `.ssh` sub-directory of your user home directory. Here it is again: 

In [44]:
def create_keypair(key_dir: Path, key_name: str) -> None:
    """A function that creates an ed25519 keypair and saves both keys
    in the directory key_dir with the stem specified by key_name. The
    secret key will not be protected by a passphrase.
    """
    passphrase = ""
    secret_key_path = key_dir / key_name

    if secret_key_file.exists():
        print(f"The file {str(secret_key_path)} already exists.")
        print("Change the value for key_name and try again.")
    else:
        cp = run(
            [
                "ssh-keygen",
                "-f",
                str(secret_key_file),
                "-N",
                passphrase,
                "-C",
                key_name,
                "-q",
            ]
        )
        if cp.returncode == 0:
            print(f"Keys were created in {str(key_dir)}.")
    return

We can use this function to create some keys to experiment with. To avoid causing any problems with keys you actually use, create keys in a new directory called `play_keys` and use `temp` as the value for `key_name`. 

In [77]:
key_name = "temp_key"
play_dir = Path.home() / "play_keys"

play_dir.mkdir(exist_ok=True)

secret_key_file =  play_dir / key_name 
public_key_file = secret_key_file.with_suffix(".pub")

if secret_key_file.exists():
    secret_key_file.unlink()
    public_key_file.unlink()
    

create_keypair(play_dir, key_name)

Keys were created in /Users/pr/play_keys.


Recall that by using an exclamation mark as the prefix for a command, you can run shell commands in the code cells of a notebook. 

Let's start by using a Python command to change to the directory `play_dir`. Then we can use `!` to run some shell commands. The first checks to see what files are in the directory.

In [79]:
os.chdir(play_dir)
!ls -la

total 24
drwxr-xr-x    5 pr  staff   160 Mar 24 16:13 .
drwxr-x---+ 134 pr  staff  4288 Mar 24 15:48 ..
-rwxr-xr-x    1 pr  staff    17 Mar 24 14:57 hello_script.sh
-rw-------    1 pr  staff   399 Mar 24 16:13 temp_key
-rw-r--r--    1 pr  staff    90 Mar 24 16:13 temp_key.pub


## Permissions 

The first two lines of output that you see when you run `ls -la` refer to directories. The one with the single `.` is the current directory, `play_dir`. The one marked with `..` is the parent of the current directory. The first letter on the lines for these two entries is `d` for directory. 

The other entries are file. They have a a dash, `-` as the first entry. The first file is the one with the secret key, `temp_keys`. The one with the public key is `temp_keys.pub`. 

After the initial `d` or `-`, there is a group of 9 characters that specify the permissions for the file or directory: 

- `rwxr-xr-x` for `.`
- `rwxr-x---` for `..`
- `rw-------` for temp_key
- `rw-r--r--` for temp_key.pub

You should think of them as three groups as three:

- `rwx r-x r-x` for `.`
- `rwx r-x ---` for `..`
- `rw- --- ---` for temp_key
- `rw- r-- r--` for temp_key.pub

The first group tells you about the permissions for the owner of this file or directory: 

- `rwx` means `can read, can write, can execute`
- `rw-` means `can read, can write, can't execute`
- `r--` means `can read, can't write, can't execute`
-  `---` means `no permission to read, write, or execute`


Here are the permissions that `ssh-keygen` gave to the secret key: 

- `rw- --- ---` for `temp_key`

This means that only the owner can read and write to this file. In contrast, the permissions for the public key, `temp_key.pub`, are:

- `rw- r-- r--` for `temp_key.pub`

These permissions let the owner of the public key read from and write to the file but in addition, it lets anyone who is in the staff group or any other user read it too. 


## Execute Permissions 

Execute permissions make sense only for a file that has an expression that is executable. Let's make a one by having code create some code. 

Usually, the `echo` command prints a string as output that is displayed on the device called `stdout`. Usually, this is a display that the user can see: 

In [59]:
!echo "echo Hello" 

echo Hello


But we can modify the command by using the `>` operator to send the output to a file: 

In [80]:
!echo "echo Hello World" > hello_script.sh

We can confirm that this command created a new file called `hello_script.sh`. 

We can use a shell command to see that this file contains a command that the shell can execute.  

In [81]:
!ls -la

total 24
drwxr-xr-x    5 pr  staff   160 Mar 24 16:13 .
drwxr-x---+ 134 pr  staff  4288 Mar 24 15:48 ..
-rwxr-xr-x    1 pr  staff    17 Mar 24 16:27 hello_script.sh
-rw-------    1 pr  staff   399 Mar 24 16:13 temp_key
-rw-r--r--    1 pr  staff    90 Mar 24 16:13 temp_key.pub


We can also use the `cat` command to see what is inside the file: 

In [82]:
!cat hello_script.sh

echo Hello World


Now let&rsquo;s use the `+x` option of the `chmod` command to give everyone execute privileges: 

In [83]:
!chmod +x hello_script.sh
!ls -la 

total 24
drwxr-xr-x    5 pr  staff   160 Mar 24 16:13 .
drwxr-x---+ 134 pr  staff  4288 Mar 24 15:48 ..
-rwxr-xr-x    1 pr  staff    17 Mar 24 16:27 hello_script.sh
-rw-------    1 pr  staff   399 Mar 24 16:13 temp_key
-rw-r--r--    1 pr  staff    90 Mar 24 16:13 temp_key.pub


Now, the owner, the group members, and all the others with accounts on the computer have an `x` in their privileges triple for `hello_script.sh`. They can all run this file. 

Run it yourself to confirm that you can. 

In the shell, the way to run an executable file is to enter the full path to the file. In this case, the full path is `./hello_script.sh` because `.` means the current directory.

In [84]:
!./hello_script.sh

Hello World


Suppose you don't want everyone to be able to execute this file. You can take execute permissions away from others using the string `o-x`: 

In [85]:
!chmod o-x hello_script.sh
!ls -la

total 24
drwxr-xr-x    5 pr  staff   160 Mar 24 16:13 .
drwxr-x---+ 134 pr  staff  4288 Mar 24 15:48 ..
-rwxr-xr--    1 pr  staff    17 Mar 24 16:27 hello_script.sh
-rw-------    1 pr  staff   399 Mar 24 16:13 temp_key
-rw-r--r--    1 pr  staff    90 Mar 24 16:13 temp_key.pub


The `o` stands for others and `-x` means &ldquo;take away execute permissions.&rdquo; 

You can also use `g` to change group permissions and `u` to change user (i.e. owner) permissions. 

For the file `temp_key.pub, here's how you could add write privileges for the group: 

In [86]:
!chmod g+w temp_key.pub 
!ls -la

total 24
drwxr-xr-x    5 pr  staff   160 Mar 24 16:13 .
drwxr-x---+ 134 pr  staff  4288 Mar 24 15:48 ..
-rwxr-xr--    1 pr  staff    17 Mar 24 16:27 hello_script.sh
-rw-------    1 pr  staff   399 Mar 24 16:13 temp_key
-rw-rw-r--    1 pr  staff    90 Mar 24 16:13 temp_key.pub


## Protecting the Secret Key

By now, you should have a hunch about one way to hardening ssh is to change the permissions on the secret key. That will turn out to be true, but to fill in the details, we must first take a detour and discuss the `sudo` command. So please read `sudo_security.ipynb` next. 
